# 03.6 — The augmentation per-call contract

Data augmentation — adding controlled noise to each trial so the model generalizes — hides the subtlest parity trap in the entire pipeline. Get it wrong and nothing crashes, no test fails loudly, and the model just quietly trains as if augmentation were off. This notebook is about the ONE rule that prevents it: **augmentation must re-randomize on every single `__getitem__`, never once per epoch.**

This notebook covers:

* What the augmentation adds (channel offset, white noise, random walk, time shift).
* The silent-parity trap: snapshot-per-epoch vs redraw-per-access.
* How `fileDatastore`'s `ReadFcn` gets this for free — and why the Python port must work for it.
* The live-read schedule contract: how curriculum magnitudes reach the dataset.

**Prerequisites:** [03.1 Dataset](03.1_dataset_vs_filedatastore.ipynb), [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb) (RNG seeding intuition). This is the capstone of Module 03.

## Section 1 — What MATLAB does

In `cgg_procAutoEncoder.m`, augmentation lives inside the datastore's read function:

```matlab
Data_Fun_Augmented = @(x) cgg_loadDataArray(x, DataWidth, ..., ...
    'STDChannelOffset', STDChannelOffset, 'STDWhiteNoise', STDWhiteNoise, ...
    'STDRandomWalk', STDRandomWalk, 'STDTimeShift', STDTimeShift, ...);
DataStore_Training.UnderlyingDatastores{1}.ReadFcn = Data_Fun_Augmented;
```

Here's the crucial, easy-to-miss part: a `fileDatastore` calls its `ReadFcn` **fresh every time a trial is read**. Inside `cgg_loadDataArray`, `cgg_generateDataAugmentationSignal` draws new random noise on each call (`randn(...)` runs every read). So a trial read in epoch 1 and again in epoch 5 gets **different** noise — MATLAB gets per-access re-randomization for free, as a side effect of how datastores work.

The Python port has to reproduce that behavior *deliberately*, because a `Dataset` does not automatically re-run anything.

## Section 2 — The Python concepts you need

### 2.1 — What the augmentation is

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from neural_data_decoding.data.augmentation import additive_augmentation_signal

rng = np.random.default_rng(0)
sig = additive_augmentation_signal(
    shape=(3, 200, 1),            # (channels, samples, areas)
    std_channel_offset=0.3,       # per-channel constant shift
    std_white_noise=0.05,         # independent per-sample noise
    std_random_walk=0.02,         # smooth low-frequency drift
    rng=rng,
)
fig, ax = plt.subplots(figsize=(9, 3.2))
for c in range(3):
    ax.plot(sig[c, :, 0], label=f"channel {c}")
ax.set_title("one augmentation draw = channel offset + white noise + random walk")
ax.set_xlabel("sample"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Three additive components (a fourth, **time shift**, jitters the window during extraction). Each is scaled by an `std_*` magnitude — and those magnitudes are what the curriculum ([Module 07](../README.md)) ramps over training. The augmented trial is `clean + this_signal`.

### 2.2 — The trap: when is the noise drawn?

The signal above changes every call — good. The trap is *where you call it*. Consider two ways to wire augmentation into a dataset:

In [ ]:
from torch.utils.data import Dataset
import torch

clean_trial = torch.zeros(4, 20)     # a stand-in "trial" (zeros so noise is all we see)

class SnapshotDataset(Dataset):
    """WRONG: draws noise ONCE, reuses it for every access of a trial."""
    def __init__(self):
        g = np.random.default_rng(0)
        self._noise = torch.tensor(g.normal(0, 0.1, (4, 20)))   # drawn ONCE, in __init__
    def __len__(self): return 1
    def __getitem__(self, i): return clean_trial + self._noise   # same noise forever

class RedrawDataset(Dataset):
    """RIGHT: draws fresh noise on every __getitem__ (the fileDatastore contract)."""
    def __init__(self):
        self._rng = np.random.default_rng(0)
    def __len__(self): return 1
    def __getitem__(self, i):
        noise = torch.tensor(self._rng.normal(0, 0.1, (4, 20)))  # drawn PER ACCESS
        return clean_trial + noise

snap, redraw = SnapshotDataset(), RedrawDataset()
print("Snapshot: access 0 == access 1 ?", torch.equal(snap[0], snap[0]))     # True — SAME noise
print("Redraw:   access 0 == access 1 ?", torch.equal(redraw[0], redraw[0])) # False — fresh noise

`SnapshotDataset` gives the model the *same* noisy trial every epoch — which is just a differently-fixed dataset, not augmentation. The model can memorize the fixed noise. `RedrawDataset` gives genuinely new noise each pass, which is what regularizes.

**Why this is a silent-parity trap specifically:** both versions run, both produce noisy trials, both let training converge, and no shape or type check catches the difference. The only symptom is worse generalization than MATLAB — a needle you'd never find without knowing to look. Critical Note #7/#8 in the migration plan exist precisely to flag it.

### 2.3 — Visualizing the difference over "epochs"

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for epoch in range(4):
    a0.plot(snap[0][0].numpy(),   alpha=0.7, label=f"epoch {epoch}")
    a1.plot(redraw[0][0].numpy(), alpha=0.7, label=f"epoch {epoch}")
a0.set_title("Snapshot: 4 'epochs' — identical (lines overlap)")
a1.set_title("Redraw: 4 'epochs' — different each time")
a0.set_xlabel("sample"); a1.set_xlabel("sample"); a1.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Left: one curve visible (all epochs identical). Right: four distinct curves.")

### 2.4 — The live-read schedule contract

There's a second, related subtlety. The augmentation *magnitudes* aren't constant — the curriculum ramps them over training (`std_white_noise` might grow from 0 to its full value across early epochs). So the dataset must read the CURRENT magnitude at access time, not a value snapshotted when it was built.

`SyntheticTrialDataset` and `MatFileTrialDataset` both hold a **reference** to the live schedule and query it inside `__getitem__` — so `schedule.update(epoch)` in the training loop is reflected on the very next batch, no dataset rebuild. Snapshot the magnitude at construction and the curriculum silently does nothing — the same class of bug as §2.2, one level up. The rule generalizes: **anything the curriculum controls must be read live, at `__getitem__` time.**

## Section 3 — The `neural_data_decoding` implementation

`MatFileTrialDataset.__getitem__`'s augmentation path — the redraw contract in production:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/mat_dataset.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip().startswith("def _apply_augmentation"))
for k in range(i, min(i + 26, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* Magnitudes come from `self.load_schedule` and are read **inside** this per-access method — the live-read contract of §2.4 (Critical Note #8, cited in the code).
* `additive_augmentation_signal` is called with the dataset's own `self._aug_rng`, which advances every call — so successive accesses get fresh noise (§2.2's redraw), never a snapshot.
* The augmentation is applied only when a schedule is present (`load_schedule is not None`) — val/test datasets pass `None` and stay clean, so metrics reflect the model, not the noise. That gate is why the training loop wires the schedule into the train dataset ONLY.
* The `_aug_rng` is independent of the dataset's data RNG, so reseeding one never perturbs the other — clean reproducibility ([00.8 §5](../00_orientation/00.8_build_a_dl_project_from_scratch.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — detect the trap

Write a function `is_augmenting(ds)` that returns True only if a dataset genuinely re-randomizes — i.e., two accesses of index 0 differ. Run it on the §2.2 snapshot and redraw datasets.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
def is_augmenting(ds):
    return not torch.equal(ds[0], ds[0])      # two accesses; differ ⇒ live augmentation

print("SnapshotDataset augmenting?", is_augmenting(snap))     # False — the trap
print("RedrawDataset augmenting?  ", is_augmenting(redraw))   # True  — correct

(This is the essence of the real property test in `tests/unit/test_augmentation.py` — assert that two reads of the same trial differ when a schedule is active.)

### Exercise 4.2 — reproducibility within the redraw

Redraw datasets still must be *reproducible across runs* (same seed → same sequence of draws). Show that two fresh `RedrawDataset()` instances produce the identical first-two-access sequence.

In [ ]:
# Reveal:
a, b = RedrawDataset(), RedrawDataset()
seq_a = [a[0].clone(), a[0].clone()]
seq_b = [b[0].clone(), b[0].clone()]
print("within a run, accesses differ:", not torch.equal(seq_a[0], seq_a[1]))
print("across instances, sequences match:",
      torch.equal(seq_a[0], seq_b[0]) and torch.equal(seq_a[1], seq_b[1]))
print("→ fresh noise each access, yet fully reproducible given the seed.")

## Section 5 — Common errors

### Model generalizes worse than the MATLAB version, no obvious cause

The augmentation snapshot trap (§2.2) — noise drawn once instead of per-access. The tell: assert two reads of the same trial differ (Exercise 4.1). This is the single most likely culprit when a faithful-looking port underperforms.

### Curriculum ramps have no effect

The live-read trap (§2.4) — the dataset snapshotted the magnitude at construction instead of reading the schedule per access. Check that `__getitem__` queries `self.load_schedule`, not a cached number.

### Validation metrics are noisy / augmented

The val/test dataset was built with a schedule. Augmentation belongs to training only — pass `load_schedule=None` for evaluation datasets.

### `num_workers > 0` changes augmentation results

Each worker gets its own RNG state ([03.2 §2.4](03.2_dataloader_and_collation.ipynb)), so the noise sequence differs from single-process. Usually fine (still valid augmentation), but if you need worker-invariant reproducibility, seed via `worker_init_fn`.

### Augmentation "works" but the signal looks wrong (discontinuous jumps)

The time-shift component indexes into the source signal; too large an `STDTimeShift` relative to the window can walk off the available samples. The loader clamps, but extreme values distort — sanity-check the magnitude against `DataWidth`.

## Section 6 — Further reading

- [Migration plan Critical Notes #7 & #8](../../docs/PLAN.md) — the canonical statement of this trap, why it's silent, and the per-call contract.
- [`src/neural_data_decoding/data/augmentation.py`](../../src/neural_data_decoding/data/augmentation.py) — the four augmentation kernels.
- [`tests/unit/test_augmentation.py`](../../tests/unit/test_augmentation.py) — the property tests that guard the redraw + live-read contracts.

**Module 03 is complete.** You've walked the full data path — files → dataset → batches → folds → normalization → augmentation — the way the production pipeline does. Module 04 (architecture) is where those batches meet the model. The [curriculum map](../README.md) tracks what's available next.